# Supervisor-confirmed robustness, association and final analytical evidence

**Purpose.** This notebook consolidates the final year-controlled prediction results with preserved horizon, linkage, nonlinear and association evidence. It also reproduces the updated lexical uncertainty and related-version sensitivity.

**Principal inputs.** All frozen v1 evidence plus `supervisor_confirmed_update_v1` and the unchanged protected folds.

**Main outputs.** Updated model hierarchy, cluster-bootstrap intervals, temporal evidence, related-version sensitivity and thesis-facing conclusions. The association models remain descriptive and non-causal.


In [ ]:
from pathlib import Path
import json
import math
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

# Find the repository from the current working directory so the notebook
# remains portable when the project folder is moved or opened in Colab.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
ROOT = next((p for p in candidates if (p / "scripts/thesis_utils_v2.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from within the Spotify_clv project.")
sys.path.insert(0, str(ROOT / "scripts"))

from thesis_utils_v2 import (
    SEED, assert_close, assert_unique_tracks, blocked_predictor_hits,
    concise_table, environment_table, extract_spotify_id, load_json,
    read_chart_archive, record_values, regression_metrics,
    save_and_show_figure, setup_notebook,
)

ROOT, FIGURE_DIR, RECORD_DIR = setup_notebook("06_robustness_inference_and_final_results.ipynb")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
display(environment_table())
print(f"Random seed: {SEED}")


## Robustness evidence matrix

The matrix below consolidates analyses already validated in their versioned tasks. It is **validated evidence loaded for synthesis**, not a new model search. Each row distinguishes supporting evidence from qualifications and supplies restrained thesis wording.


In [ ]:
robustness = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/final_robustness_evidence_matrix.csv"
)
display(robustness[[
    "claim_or_issue", "supporting_evidence", "contrary_or_mixed_evidence",
    "evidence_strength", "safe_thesis_wording",
]])

year_metrics = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/year_controlled_grouped_cv_metrics.csv"
)
year_temporal = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/year_controlled_temporal_metrics.csv"
)
lz_summary = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/L_Z_descriptives.csv"
)
cohort_bridge = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/survival1357_to_y52_1014_summary.csv"
)
display(year_metrics[["model_id", "n", "mae", "rmse", "mean_prediction_bias"]].round(3))
display(lz_summary.round(4))
display(cohort_bridge)

final_comparison = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/thesis_final_model_comparison.csv"
)
historical_roles = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/final_candidate_model_roles.csv"
)
historical_findings = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/final_finding_hierarchy.csv"
)


## Cluster-bootstrap uncertainty

The paired bootstrap resamples the 276 protected artist/version components rather than individual songs. It quantifies sampling uncertainty in the observed error difference under that grouping structure. It does not prove causality, correct all selection bias, or supply a Bayesian posterior probability.


In [ ]:
paired = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/year_controlled_lexical_uncertainty.csv"
)
display(paired[[
    "reference_model", "candidate_model", "metric", "observed_difference",
    "ci_2_5", "ci_97_5", "bootstrap_proportion_candidate_better", "n_clusters",
]].round(4))

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.errorbar(
    paired.observed_difference, paired.metric,
    xerr=[paired.observed_difference - paired.ci_2_5,
          paired.ci_97_5 - paired.observed_difference],
    fmt="o", capsize=4, color="#4C78A8"
)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set(title="LEX_YEAR improvement over F3_YEAR",
       xlabel="Reference error minus candidate error (weeks)", ylabel="")
save_and_show_figure(fig, FIGURE_DIR / "nb06_year_lexical_cluster_intervals.png")


LEX_YEAR retains a small positive MAE point difference, but its 95% interval now just includes zero; the RMSE interval also includes zero. The protected-component bootstrap is conditional on fixed held-out predictions and does not include model-selection uncertainty. The lexical contribution therefore remains modest and qualified.


In [ ]:
import run_provisional_final_model_comparison as final_models
import run_supervisor_confirmed_update as supervisor_update

audit_rows = []
data, data26, assignments, temporal52, temporal26 = final_models.validate_inputs(audit_rows)
ols_coefficients, standardised_associations = final_models.fit_all_ols(data, audit_rows)
frozen_ols = pd.read_csv(
    ROOT / "data/processed/provisional_final_model_comparison_v1/clustered_ols_coefficients.csv"
)
key_columns = ["association_model", "clustering_unit", "term"]
aligned_ols = ols_coefficients.merge(
    frozen_ols[key_columns + ["coefficient_weeks", "cluster_robust_se"]].rename(columns={
        "coefficient_weeks": "validated_coefficient",
        "cluster_robust_se": "validated_se",
    }), on=key_columns, validate="one_to_one"
)
assert np.allclose(aligned_ols.coefficient_weeks, aligned_ols.validated_coefficient, atol=1e-10)
assert np.allclose(aligned_ols.cluster_robust_se, aligned_ols.validated_se, atol=1e-10)

primary_ols = ols_coefficients[
    ols_coefficients.clustering_unit.eq("first_credited_artist_primary")
    & ols_coefficients.term.isin(["X_level_2_vs_1", "X_level_3_vs_1", "X_level_4_vs_1"])
]
display(primary_ols[[
    "association_model", "term", "coefficient_weeks", "cluster_robust_se", "ci_2_5", "ci_97_5",
    "n_tracks", "n_clusters"
]].round(3))


update_data, update_assignments, update_temporal = supervisor_update.load_model_data()


The historical artist-clustered association models remain descriptive companions rather than principal predictive specifications. Their X=4 contrast is large across A1–A3: tracks present in all four early weeks have substantially more future chart weeks than the X=1 reference, conditional on included covariates. This is an association with observed momentum, not an intervention or causal effect; it is unchanged because the requested entry-year update concerns the principal predictive sequence.


In [ ]:
correlation_summary, pearson, spearman = final_models.final_correlation_analysis(data, audit_rows)
stability = final_models.coefficient_stability_audit(data, ols_coefficients, correlation_summary)
nb2, nb2_info = final_models.fit_nb2_association(data, standardised_associations)
display(stability.sort_values(["flag_vif_10plus", "vif"], ascending=False)[[
    "term", "source_variable", "a2_coefficient", "a3_coefficient", "vif",
    "direction_changed", "coefficient_interpretation_flag"
]].head(15).round(3))
display(nb2[nb2.term.isin(["X_level_2_vs_1", "X_level_3_vs_1", "X_level_4_vs_1"])][[
    "term", "coefficient_log_mean", "rate_ratio", "ci_2_5_log_scale", "ci_97_5_log_scale",
    "direction_agrees_with_standardised_OLS"
]].round(3))
print("NB2 converged:", nb2_info["converged"], "dispersion alpha:", nb2_info["alpha"])


NB2 supports the broad positive X pattern, but not every audio or lexical coefficient agrees in sign with standardised OLS. This reinforces the decision to interpret predictor blocks and predictive performance rather than isolated, unstable coefficients.

## H3 descriptive logistic association

The H3 Part A odds ratios describe association with any future chart presence. They complement the held-out classification metrics; they do not turn the predictive result into a causal explanation.


In [ ]:
logistic, logistic_info = final_models.fit_logistic_association(data)
display(logistic[logistic.term.isin(["X_level_2_vs_1", "X_level_3_vs_1", "X_level_4_vs_1"])][[
    "term", "coefficient_log_odds", "odds_ratio", "odds_ratio_ci_2_5", "odds_ratio_ci_97_5",
    "n_artist_clusters"
]].round(3))
print("Descriptive H3 logistic fit converged:", logistic_info["converged"])


## Conservative related-version exclusion

The primary unit remains Spotify track ID. This sensitivity removes all 143 tracks in the 70 supported exact-lyric/related-version groups, while retaining the two tracks in the unresolved group. Fold labels remain unchanged. The model refit and 5,000-replicate cluster interval are recomputed here using the validated routine.


In [ ]:
related_sensitivity, related_predictions = supervisor_update.related_version_sensitivity(update_data)
display(related_sensitivity[[
    "record_type", "model_id", "n_tracks_removed", "n_tracks_remaining", "mae", "rmse",
    "metric", "observed_difference", "ci_2_5", "ci_97_5",
]].round(4))

frozen_related = pd.read_csv(
    ROOT / "data/processed/supervisor_confirmed_update_v1/year_controlled_related_version_sensitivity.csv"
)
new_mae = related_sensitivity[
    related_sensitivity.record_type.eq("paired_cluster_bootstrap")
    & related_sensitivity.metric.eq("MAE")
].iloc[0]
saved_mae = frozen_related[
    frozen_related.record_type.eq("paired_cluster_bootstrap")
    & frozen_related.metric.eq("MAE")
].iloc[0]
assert int(new_mae.n_tracks_removed) == 143
assert int(new_mae.n_tracks_remaining) == 871
assert_close(new_mae.observed_difference, saved_mae.observed_difference, 1e-10,
             "year-controlled version sensitivity MAE difference")
assert_close(new_mae.ci_2_5, saved_mae.ci_2_5, 1e-10,
             "year-controlled version sensitivity lower CI")
assert_close(new_mae.ci_97_5, saved_mae.ci_97_5, 1e-10,
             "year-controlled version sensitivity upper CI")


After the unchanged conservative exclusion, LEX_YEAR improves MAE over F3_YEAR by about 0.14 weeks, but the interval includes zero. The direction remains positive while certainty weakens, so related recording/version structure remains an explicit limitation.

## Final supervisor-confirmed model hierarchy

F1_YEAR, F3_YEAR and LEX_YEAR form the principal count sequence. H3_YEAR is the companion continuation/two-part analysis. TF-IDF, nonlinear models, alternative count forms, Y26 and 160k analyses remain sensitivity evidence rather than additions to the principal model search.


In [ ]:
roles = pd.DataFrame([
    ("F1_YEAR", "early persistence + entry year", "principal benchmark"),
    ("F3_YEAR", "F1_YEAR + audio/song descriptors", "principal comparison"),
    ("LEX_YEAR", "F3_YEAR + seven lexical summaries", "principal interpretable lyric comparison"),
    ("H3_YEAR", "two-part future presence and positive duration", "companion probability/two-part analysis"),
    ("T3", "fold-local TF-IDF", "sensitivity/appendix"),
    ("RF/ET/HGB", "preserved nonlinear comparisons", "sensitivity/appendix"),
], columns=["model_id", "content", "final_role"])
display(roles)

principal_table = year_metrics.set_index("model_id").loc[
    ["F1_YEAR", "F3_YEAR", "LEX_YEAR", "H3_YEAR"]
].reset_index()
display(principal_table[["model_id", "mae", "rmse", "mean_prediction_bias"]].round(3))

plot = principal_table.melt(
    id_vars="model_id", value_vars=["mae", "rmse"],
    var_name="metric", value_name="error"
)
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=plot, x="model_id", y="error", hue="metric",
            palette=["#4C78A8", "#F58518"], ax=ax)
ax.set(title="Final year-controlled grouped OOF evidence",
       xlabel="Model", ylabel="Prediction error (chart weeks)")
ax.legend(title="", labels=["MAE", "RMSE"])
save_and_show_figure(fig, FIGURE_DIR / "nb06_final_year_controlled_comparison.png")


## Final analytical conclusions

**Tier 1.** Among songs already in the Global Top 200, categorical early four-week chart persistence remains the dominant predictive signal after controlling for entry year. Standard audio descriptors add little beyond X. Predicting any continuation is easier than predicting exact positive longevity.

**Tier 2.** Seven transparent lexical summaries provide a small grouped-CV MAE increment. The year-controlled interval and the related-version sensitivity both require restrained wording.

**Tier 3.** TF-IDF, nonlinear ML, temporal lyric effects, Y26 and alternative cohort/model-form results remain mixed or sensitivity evidence.

### Limitations

The study is conditional on chart entry and is not hit prediction. Exact-ID feature coverage is selective, complete collaborator separation is not claimed, three source weeks remain unknown, and the Y52 temporal 2020 sample contains only 45 tracks. L and Z describe the outcome window and cannot be used as predictors. Association coefficients are non-causal.

### Safe claims and claims to avoid

It is safe to state that X supplies most of the out-of-sample signal, audio adds little after X, and continuation probability is easier to estimate than exact positive duration. The lexical increment should be called small or modest. The evidence does not support causal claims, a strong temporal lyric advantage, or promotion of TF-IDF/nonlinear sensitivities to the principal sequence.


In [ ]:
a1_x4 = ols_coefficients[
    ols_coefficients.association_model.eq("A1")
    & ols_coefficients.clustering_unit.eq("first_credited_artist_primary")
    & ols_coefficients.term.eq("X_level_4_vs_1")
].iloc[0]
lex_2019 = year_temporal[
    year_temporal.temporal_partition.eq("validation_2019")
    & year_temporal.model_id.eq("LEX_YEAR")
].iloc[0]
lex_2020 = year_temporal[
    year_temporal.temporal_partition.eq("test_2020")
    & year_temporal.model_id.eq("LEX_YEAR")
].iloc[0]
lexical_mae = paired[paired.metric.eq("MAE")].iloc[0]
lz_values = lz_summary.set_index("measure").value
record_values(RECORD_DIR / "nb06_values.json", "06_robustness_inference_and_final_results", {
    "ols_a1_x4_coefficient": a1_x4.coefficient_weeks,
    "ols_a1_x4_ci_low": a1_x4.ci_2_5,
    "ols_a1_x4_ci_high": a1_x4.ci_97_5,
    "version_removed_n": int(new_mae.n_tracks_removed),
    "version_remaining_n": int(new_mae.n_tracks_remaining),
    "version_lex_year_mae_difference": new_mae.observed_difference,
    "version_lex_year_mae_ci_low": new_mae.ci_2_5,
    "version_lex_year_mae_ci_high": new_mae.ci_97_5,
    "lex_year_2019_mae": lex_2019.mae,
    "lex_year_2019_rmse": lex_2019.rmse,
    "lex_year_2020_n": int(lex_2020.n),
    "lex_year_2020_mae": lex_2020.mae,
    "y52_l52_pearson": lz_values["Y52_L52_Pearson"],
    "y52_l52_spearman": lz_values["Y52_L52_Spearman"],
    "z_multiple_blocks_percent": lz_values["Z_greater_than_1_percent"],
    "f3_year_lex_year_mae_difference": lexical_mae.observed_difference,
    "nb2_converged_historical": bool(nb2_info["converged"]),
})
print("Notebook 06 supervisor-confirmed reconciliation record saved.")
